# 05 · Selección y política de alertas

El mayor ROC-AUC medio de validación determina el champion. Random Forest quedó primero en la comparación original y en la revisión de configuraciones por defecto. La comparación actual aplica también el criterio de mejora del tuning, antes de fijar los roles.

La grilla contiene umbrales entre 0,05 y 0,95. Se selecciona la mejor alternativa con al menos una alerta en CV por diferencia media de capital frente a Buy & Hold, con recall como desempate. Si la diferencia es negativa, la política de negocio preferida en CV es Buy & Hold.

El hold-out ya se utilizó en la entrega original. Los resultados de esta revisión corresponden a una reevaluación de ese tramo.

In [ ]:
EVALUAR_HOLDOUT = True
PUBLICAR_ALIAS = False

In [12]:
import json
import os
import sys
from hashlib import sha256
from pathlib import Path
from time import perf_counter

import mlflow
import mlflow.sklearn
import pandas as pd

from mlflow import MlflowClient
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.svm import SVC

directorio = Path.cwd().resolve()
raiz = next(
    (
        ruta for ruta in [directorio, *directorio.parents]
        if (ruta / "src" / "pipeline.py").is_file()
    ),
    None,
)

if raiz is None:
    raise FileNotFoundError("Abre el notebook desde el repositorio pset3.")

sys.path.insert(0, str(raiz))

from src.datos import cargar_obt
from src.features import construir_features
from src.pipeline import crear_pipeline

SEMILLA = 42
HORIZONTE = 5
CUANTIL_TARGET = 0.80
inicio_prueba = pd.Timestamp("2023-07-11")
capital_inicial = 10_000.0

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_registry_uri("http://127.0.0.1:5000")
cliente = MlflowClient()
NOMBRE_EXPERIMENTO = "Sistema de alerta temprana de volatilidad para acciones"
NOMBRE_REGISTRO = "alerta-volatilidad-aapl-modelo"
ruta_revision = raiz / "docs" / "revision_runs.json"

def guardar_ids(contenido):
    temporal = ruta_revision.with_suffix(".tmp")
    ruta_revision.parent.mkdir(exist_ok=True)
    temporal.write_text(json.dumps(contenido, indent=2), encoding="utf-8")
    temporal.replace(ruta_revision)

revision = (
    json.loads(ruta_revision.read_text(encoding="utf-8"))
    if ruta_revision.is_file() else {}
)

In [13]:
df = cargar_obt()
snapshot = df.sort_values(["TRADE_DATE", "SYMBOL"])

datos_corrida = {
    "tabla_origen": "PSET2_DB.OBT.AAPL_DAILY_OBT",
    "fecha_corte_snapshot": os.environ["DATA_CUTOFF"],
    "sha256_snapshot_obt": sha256(
        snapshot.to_csv(index=False, lineterminator="\n").encode("utf-8")
    ).hexdigest(),
    "filas_snapshot": len(snapshot),
}

print("Dimensiones de la OBT:", df.shape)
print(datos_corrida)

Dimensiones de la OBT: (4087, 19)
{'tabla_origen': 'PSET2_DB.OBT.AAPL_DAILY_OBT', 'fecha_corte_snapshot': '2026-09-30', 'sha256_snapshot_obt': '673213bc5b0fe2ef795cd2492ece28723d4b3d92656aadfb5cadb71ed13dbafb', 'filas_snapshot': 4087}


In [14]:
features = construir_features(df)

vol_futura_5 = (
    features["RET_LAG_0"]
    .rolling(HORIZONTE, min_periods=HORIZONTE)
    .std(ddof=1)
    .shift(-HORIZONTE)
    .rename("VOL_FUTURA_5")
)

fin_ventana_target = features.index.to_series().shift(-HORIZONTE)

datos_modelado = (
    features.iloc[60:]
    .join(vol_futura_5)
    .dropna(subset=["VOL_FUTURA_5"])
)

entrenamiento = datos_modelado.loc[
    fin_ventana_target.reindex(datos_modelado.index) < inicio_prueba
].copy()

fechas_holdout = datos_modelado.index[datos_modelado.index >= inicio_prueba]

X_desarrollo = entrenamiento.drop(columns="VOL_FUTURA_5")
vol_desarrollo = entrenamiento["VOL_FUTURA_5"]
columnas_numericas = X_desarrollo.columns.drop("MES").tolist()
cv_temporal = TimeSeriesSplit(n_splits=5, gap=HORIZONTE)

hash_desarrollo = sha256(
    entrenamiento.to_csv(index=True, lineterminator="\n").encode("utf-8")
).hexdigest()

# Retorno de t a t+1, expresado como proporción y alineado con t.
retorno_siguiente = features["RET_LAG_0"].shift(-1).div(100)

assert fin_ventana_target.loc[X_desarrollo.index].max() < fechas_holdout.min()

print("Desarrollo:", X_desarrollo.shape)
print("Fechas reservadas para prueba:", len(fechas_holdout))

Desarrollo: (3212, 18)
Fechas reservadas para prueba: 805


In [15]:
tuning_disponible = (
    revision.get("sha256_dataset_desarrollo") == hash_desarrollo
    and len(revision.get("tuning", {})) == 3
)
if not tuning_disponible:
    print("Ejecuta 03 y 04 con esta extracción para preparar la selección.")
if PUBLICAR_ALIAS and not (tuning_disponible and EVALUAR_HOLDOUT):
    raise ValueError("La publicación requiere tuning vigente y evaluación del hold-out.")

In [16]:
if tuning_disponible:
    experimento = cliente.get_experiment_by_name(NOMBRE_EXPERIMENTO)
    estimadores_disponibles = {
        "RandomForestClassifier": ("Random Forest", RandomForestClassifier(random_state=SEMILLA)),
        "LogisticRegression": ("Regresión logística", LogisticRegression(random_state=SEMILLA)),
        "SVC": ("SVM", SVC(probability=True, random_state=SEMILLA)),
    }
    filas = []
    estimadores_finalistas = {}

    for clase, run_id in revision["tuning"].items():
        nombre, estimador = estimadores_disponibles[clase]
        run = cliente.get_run(run_id)
        parametros, metricas, tags = run.data.params, run.data.metrics, run.data.tags
        assert run.info.status == "FINISHED"
        assert run.info.experiment_id == experimento.experiment_id
        assert parametros["sha256_dataset_desarrollo"] == hash_desarrollo
        assert parametros["run_id_default"] in revision["candidatos"]
        assert tags.get("fase") == "tuning" and tags.get("mejor_run_hijo")
        assert tags["mejora_segun_pset"] in ("true", "false")

        ajustado = tags["mejora_segun_pset"] == "true"
        if ajustado:
            archivo = mlflow.artifacts.download_artifacts(
                artifact_uri=f"runs:/{run_id}/mejores_parametros.json"
            )
            estimador.set_params(**json.loads(Path(archivo).read_text(encoding="utf-8")))

        estimadores_finalistas[nombre] = estimador
        filas.append({
            "modelo": nombre,
            "version_elegida": "ajustado" if ajustado else "default",
            "run_id_elegido": tags["mejor_run_hijo"] if ajustado else parametros["run_id_default"],
            "auc_elegido": metricas["roc_auc_val_media_cv" if ajustado else "roc_auc_default"],
            "std_elegida": metricas["roc_auc_val_std_cv" if ajustado else "roc_auc_default_std_cv"],
        })

    comparacion = pd.DataFrame(filas).sort_values(
        ["auc_elegido", "std_elegida", "modelo"], ascending=[False, True, True]
    )
    finalistas = comparacion.head(2).copy()
    finalistas.insert(0, "alias", ["champion", "challenger"])
    alias_por_modelo = finalistas.set_index("modelo")["alias"].to_dict()
    display(comparacion.round(4))
    display(finalistas.round(4))

,modelo,version_elegida,run_id_elegido,auc_elegido,std_elegida
0,Random Forest,default,432e8271a62e42389de9d59e8d934d48,0.7226,0.0480
1,Regresión logística,default,f7e8fd5b27da4cc890134287aae13786,0.7099,0.0679
2,SVM,default,b6350a1f8f5945668664408312e7d68f,0.6995,0.0401


,alias,modelo,version_elegida,run_id_elegido,auc_elegido,std_elegida
0,champion,Random Forest,default,432e8271a62e42389de9d59e8d934d48,0.7226,0.0480
1,challenger,Regresión logística,default,f7e8fd5b27da4cc890134287aae13786,0.7099,0.0679


In [17]:
if tuning_disponible:
    predicciones_cv = {}

    for nombre in finalistas["modelo"]:
        bloques = []

        for pliegue, (idx_train, idx_val) in enumerate(
            cv_temporal.split(X_desarrollo), start=1
        ):
            X_train = X_desarrollo.iloc[idx_train]
            X_val = X_desarrollo.iloc[idx_val]
            vol_train = vol_desarrollo.iloc[idx_train]
            vol_val = vol_desarrollo.iloc[idx_val]

            p80 = vol_train.quantile(CUANTIL_TARGET)
            y_train = vol_train.gt(p80).astype(int)
            y_val = vol_val.gt(p80).astype(int)

            pipeline = crear_pipeline(
                clone(estimadores_finalistas[nombre]),
                columnas_numericas,
            )
            pipeline.fit(X_train, y_train)
            proba_alta = pipeline.predict_proba(X_val)[:, 1]

            bloques.append(pd.DataFrame({
                "pliegue": pliegue,
                "clase_alta": y_val,
                "proba_alta": proba_alta,
                "retorno_siguiente": retorno_siguiente.loc[X_val.index],
            }, index=X_val.index))

        predicciones_cv[nombre] = pd.concat(bloques).sort_index()
        print(f"{nombre}: {len(predicciones_cv[nombre]):,} predicciones de validación")

Random Forest: 2,675 predicciones de validación
Regresión logística: 2,675 predicciones de validación


In [18]:
if tuning_disponible:
    # Experimentando con el umbral

    import numpy as np
    from sklearn.metrics import recall_score

    EXPOSICION_CON_ALERTA = 0.75
    COSTO_TRANSACCION = 0.0
    umbrales = np.linspace(0.05, 0.95, 19)


    def simular_estrategia(retornos, alertas):
        alerta_vigente = alertas.rolling(HORIZONTE, min_periods=1).max().astype(bool)
        exposicion = pd.Series(1.0, index=retornos.index).mask(
            alerta_vigente, EXPOSICION_CON_ALERTA
        )
        exposicion_previa = exposicion.shift(1, fill_value=1.0)
        costo = exposicion.sub(exposicion_previa).abs() * COSTO_TRANSACCION
        retorno_neto = exposicion.mul(retornos).sub(costo)
        return capital_inicial * retorno_neto.add(1).cumprod()


    def drawdown_cv_pct(capital):
        maximos = capital.cummax().clip(lower=capital_inicial)
        return float(-100 * capital.div(maximos).sub(1).min())


    resultados_umbrales = []

    for nombre, predicciones in predicciones_cv.items():
        for umbral in umbrales:
            filas_pliegues = []

            for _, bloque in predicciones.groupby("pliegue", sort=True):
                retornos = bloque["retorno_siguiente"]
                alertas = bloque["proba_alta"].ge(umbral)

                capital_alertas = simular_estrategia(retornos, alertas)
                capital_buy_hold = capital_inicial * retornos.add(1).cumprod()

                filas_pliegues.append({
                    "alertas_cv": int(alertas.sum()),
                    "diferencia_capital_cv_usd": (
                        capital_alertas.iloc[-1] - capital_buy_hold.iloc[-1]
                    ),
                    "recall_cv": recall_score(
                        bloque["clase_alta"], alertas, zero_division=0
                    ),
                    "mdd_cv_pct": drawdown_cv_pct(capital_alertas),
                    "mdd_buy_hold_cv_pct": drawdown_cv_pct(capital_buy_hold),
                })

            resumen = pd.DataFrame(filas_pliegues)
            resultados_umbrales.append({
                "modelo": nombre,
                "umbral_alerta": float(umbral),
                "alertas_cv": int(resumen["alertas_cv"].sum()),
                "tasa_alertas_cv_pct": (
                    100 * resumen["alertas_cv"].sum() / len(predicciones)
                ),
                "diferencia_capital_cv_usd": resumen["diferencia_capital_cv_usd"].mean(),
                "recall_cv": resumen["recall_cv"].mean(),
                "mdd_cv_pct": resumen["mdd_cv_pct"].mean(),
                "mdd_buy_hold_cv_pct": resumen["mdd_buy_hold_cv_pct"].mean(),
            })

    tabla_umbrales = pd.DataFrame(resultados_umbrales)

    umbrales_elegidos = (
        tabla_umbrales.loc[tabla_umbrales["alertas_cv"].gt(0)]
        .sort_values(
            ["diferencia_capital_cv_usd", "recall_cv", "umbral_alerta"],
            ascending=[False, False, False],
        )
        .drop_duplicates("modelo")
        .reset_index(drop=True)
    )

    assert set(umbrales_elegidos["modelo"]) == set(predicciones_cv), (
        "Al menos un modelo no generó alertas con los umbrales evaluados."
    )

    display(umbrales_elegidos.round(4))

    if umbrales_elegidos["diferencia_capital_cv_usd"].le(0).all():
        print("Ningún umbral activo probado supera a Buy & Hold en capital final medio.")

,modelo,umbral_alerta,alertas_cv,tasa_alertas_cv_pct,diferencia_capital_cv_usd,recall_cv,mdd_cv_pct,mdd_buy_hold_cv_pct
0,Random Forest,0.90,1,0.0374,-81.3027,0.0019,32.9339,32.9339
1,Regresión logística,0.95,3,0.1121,-154.7937,0.0018,32.9605,32.9339


Ningún umbral activo probado supera a Buy & Hold en capital final medio.


## Reevaluación del hold-out

Los roles y umbrales se fijan en las celdas anteriores.

In [19]:
if tuning_disponible and EVALUAR_HOLDOUT:
    X_test = datos_modelado.loc[fechas_holdout, X_desarrollo.columns]
    assert X_test.shape == (805, 18)

    p80_final = float(vol_desarrollo.quantile(CUANTIL_TARGET))
    y_desarrollo_final = vol_desarrollo.gt(p80_final).astype(int)
    y_test = datos_modelado.loc[fechas_holdout, "VOL_FUTURA_5"].gt(p80_final).astype(int)

    retornos_test = retorno_siguiente.loc[fechas_holdout]
    assert retornos_test.notna().all() and y_test.nunique() == 2

    umbrales_por_modelo = umbrales_elegidos.set_index("modelo")["umbral_alerta"]
    capital_buy_hold = capital_inicial * retornos_test.add(1).cumprod()


    def max_drawdown_pct(capital):
        maximos = capital.cummax().clip(lower=capital_inicial)
        return float(-100 * capital.div(maximos).sub(1).min())


    modelos_finales = {}
    curvas_capital = {"Buy & Hold": capital_buy_hold}
    resultados_finales = []

    for fila in finalistas.itertuples(index=False):
        nombre = fila.modelo
        umbral = float(umbrales_por_modelo.loc[nombre])
        pipeline = crear_pipeline(
            clone(estimadores_finalistas[nombre]),
            columnas_numericas,
        )

        inicio = perf_counter()
        pipeline.fit(X_desarrollo, y_desarrollo_final)
        segundos_ajuste = perf_counter() - inicio

        inicio = perf_counter()
        proba_alta = pd.Series(
            pipeline.predict_proba(X_test)[:, 1],
            index=fechas_holdout,
        )
        segundos_inferencia = perf_counter() - inicio

        alertas = proba_alta.ge(umbral)
        capital_modelo = simular_estrategia(retornos_test, alertas)

        modelos_finales[nombre] = pipeline
        curvas_capital[nombre] = capital_modelo

        resultados_finales.append({
            "modelo": nombre,
            "alias": alias_por_modelo[nombre],
            "run_id_candidato": fila.run_id_elegido,
            "roc_auc_cv": fila.auc_elegido,
            "std_cv": fila.std_elegida,
            "roc_auc_test": roc_auc_score(y_test, proba_alta),
            "umbral_alerta": umbral,
            "alertas_test": int(alertas.sum()),
            "recall_alertas_test": recall_score(y_test, alertas, zero_division=0),
            "capital_final_usd": float(capital_modelo.iloc[-1]),
            "diferencia_vs_buy_hold_usd": float(
                capital_modelo.iloc[-1] - capital_buy_hold.iloc[-1]
            ),
            "max_drawdown_pct": max_drawdown_pct(capital_modelo),
            "segundos_ajuste": segundos_ajuste,
            "ms_inferencia_por_fila": 1000 * segundos_inferencia / len(X_test),
        })

    tabla_final = pd.DataFrame(resultados_finales)
    display(tabla_final.round(4))

    print(f"P80 final: {p80_final:.4f} %")
    print(f"Capital final de Buy & Hold: USD {capital_buy_hold.iloc[-1]:,.2f}")
    print(f"Drawdown máximo de Buy & Hold: {max_drawdown_pct(capital_buy_hold):.2f} %")
else:
    print('Reevaluación del hold-out desactivada.')

,modelo,alias,run_id_candidato,roc_auc_cv,std_cv,roc_auc_test,umbral_alerta,alertas_test,recall_alertas_test,capital_final_usd,diferencia_vs_buy_hold_usd,max_drawdown_pct,segundos_ajuste,ms_inferencia_por_fila
0,Random Forest,champion,432e8271a62e42389de9d59e8d934d48,0.7226,0.0480,0.6356,0.90,0,0.00,18126.4947,0.0000,33.3605,3.6962,0.0340
1,Regresión logística,challenger,f7e8fd5b27da4cc890134287aae13786,0.7099,0.0679,0.6439,0.95,7,0.06,17560.8986,-565.5961,32.4870,0.0603,0.0122


P80 final: 2.1072 %
Capital final de Buy & Hold: USD 18,126.49
Drawdown máximo de Buy & Hold: 33.36 %


## Publicación en MLflow

Activar PUBLICAR_ALIAS cambia los alias que puede consumir el servicio del PSet 4.

In [22]:
if PUBLICAR_ALIAS:
    import subprocess

    from mlflow.models import infer_signature

    commit_final = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=raiz, text=True
    ).strip()

    columnas_metricas = [
        "roc_auc_cv",
        "std_cv",
        "roc_auc_test",
        "alertas_test",
        "recall_alertas_test",
        "capital_final_usd",
        "diferencia_vs_buy_hold_usd",
        "max_drawdown_pct",
        "segundos_ajuste",
        "ms_inferencia_por_fila",
    ]

    registro_final = []

    for fila in tabla_final.to_dict("records"):
        nombre = fila["modelo"]
        alias = alias_por_modelo[nombre]
        pipeline = modelos_finales[nombre]
        ejemplo = X_desarrollo.head(5).copy()

        with mlflow.start_run(
            experiment_id=experimento.experiment_id,
            run_name=f"evaluacion_final_{alias}",
        ) as corrida:
            mlflow.set_tags({
                "fase": "final",
                "modelo": nombre,
                "alias_destino": alias,
                "git_commit": commit_final,
            })
            mlflow.log_params({
                **datos_corrida,
                "sha256_dataset_desarrollo": hash_desarrollo,
                "run_id_candidato": fila["run_id_candidato"],
                "p80_final": p80_final,
                "umbral_alerta": fila["umbral_alerta"],
                "inicio_holdout": fechas_holdout[0].date().isoformat(),
                "fin_holdout": fechas_holdout[-1].date().isoformat(),
                "filas_holdout": len(fechas_holdout),
            })
            mlflow.log_metrics({
                metrica: float(fila[metrica]) for metrica in columnas_metricas
            })

            artefacto = mlflow.sklearn.log_model(
                pipeline,
                name=f"pipeline_final_{alias}",
                signature=infer_signature(ejemplo, pipeline.predict_proba(ejemplo)),
                input_example=ejemplo,
                pyfunc_predict_fn="predict_proba",
            )

            version = mlflow.register_model(artefacto.model_uri, NOMBRE_REGISTRO)
            cliente.set_model_version_tag(
                NOMBRE_REGISTRO, version.version, "run_id_final", corrida.info.run_id
            )
        

            registro_final.append({
                "modelo": nombre,
                "alias": alias,
                "version": version.version,
                "run_id_final": corrida.info.run_id,
                "roc_auc_test": fila["roc_auc_test"],
            })

    for fila in registro_final:
        cliente.set_registered_model_alias(NOMBRE_REGISTRO, fila["alias"], fila["version"])

    display(pd.DataFrame(registro_final))
else:
    print('Actualización de alias desactivada.')

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/27 19:35:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which c

Registered model 'alerta-volatilidad-aapl-modelo' already exists. Creating a new version of this model...
2026/11/27 19:39:30 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: alerta-volatilidad-aapl-modelo, version 3
Created version '3' of model 'alerta-volatilidad-aapl-modelo'.


🏃 View run evaluacion_final_champion at: http://127.0.0.1:5000/#/experiments/1/runs/cdfde8f825a141ef9fd839389c099b4f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/27 19:39:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which c

Registered model 'alerta-volatilidad-aapl-modelo' already exists. Creating a new version of this model...
2026/11/27 19:40:40 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: alerta-volatilidad-aapl-modelo, version 4
Created version '4' of model 'alerta-volatilidad-aapl-modelo'.


🏃 View run evaluacion_final_challenger at: http://127.0.0.1:5000/#/experiments/1/runs/e51139f81b334a529ac08e14cc4b71b6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


,modelo,alias,version,run_id_final,roc_auc_test
0,Random Forest,champion,3,cdfde8f825a141ef9fd839389c099b4f,0.635553
1,Regresión logística,challenger,4,e51139f81b334a529ac08e14cc4b71b6,0.643915
